In [ ]:
import os

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from jax.scipy.spatial.transform import Rotation as Rot

import genmetaballs.fmb.fm_render as fm_render
from genmetaballs.core import (
    FMB,
    Intrinsics,
    ThreeParameterBlender,
    ZeroParameterConfidence,
    geometry,
    make_fmb_scene_from_values,
    render_fmbs,
)
from genmetaballs.fmb.utils import DegradeLR, get_camera_rays, image_grid

Pose, Vec3D, Rotation = geometry.Pose, geometry.Vec3D, geometry.Rotation


# JAX setup (before importing)
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
np.random.seed(0)
print(f"JAX version: {jax.__version__}")
print(f"JAX backend: {jax.default_backend().upper()}")
print(f"JAX devices: {jax.devices()}")

In [ ]:
def visualize_results(gmb, ref, title=""):
    fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=(8, 5))
    fig.suptitle(title)
    im0 = ax0.imshow(gmb)
    ax0.set_title("GenMetaBalls", fontsize=12, fontweight="bold")
    ax0.axis("off")
    plt.colorbar(im0, ax=ax0)

    ax1.set_title("Reference Impl.", fontsize=12, fontweight="bold")
    im1 = ax1.imshow(ref)
    ax1.axis("off")
    plt.colorbar(im1, ax=ax1)

    diff = gmb - ref
    ax2.set_title("Difference", fontsize=12, fontweight="bold")
    im2 = ax2.imshow(diff)
    ax2.axis("off")
    plt.colorbar(im2, ax=ax2)
    return jnp.linalg.norm(diff)


In [ ]:
height, width = image_size = (64, 80)
focal_length = 20.0
cx = (width - 1) / 2
cy = (height - 1) / 2

In [ ]:
# integer xy-coordinates (as opposed to ij coordinates) of the pixels in the image
# augmented with zeros
pixel_list = (
    (np.array(np.meshgrid(np.arange(width), height - np.arange(height) - 1, [0]))[:, :, :, 0])
    .reshape((3, -1))
    .T
)
# camera frame in get_camera rays has -z pointing into the image!!!!
camera_rays = get_camera_rays(focal_length, focal_length, cx, cy, pixel_list)
rays_trans = np.stack([camera_rays, np.tile(np.zeros(3), (camera_rays.shape[0], 1))], 1)

In [ ]:
hyperparams = fm_render.hyperparams
beta2 = jnp.float32(np.exp(hyperparams[0]))
beta3 = jnp.float32(np.exp(hyperparams[1]))

In [ ]:
# reference impl
render_jit = jax.jit(fm_render.render_func_rays)

means = jnp.array([[-4.0, -2, -4], [3, 4, -6]])
extents = jnp.array([[55.0, 4, 1], [8.0, 6, 1]])
rots = [
    Rot.from_euler("ZYX", jnp.array([jnp.pi / 3, 0, 0])),
    Rot.from_euler("ZYX", jnp.array([-jnp.pi / (2.31), 0, 0])),
]
# means, extents, rots = means[1:2], extents[1:2], rots[1:2]
assert len(means) == len(extents) == len(rots)
N = len(means)
log_weights = jnp.zeros(N) - jnp.log(N)

In [ ]:
# reference impl
rotmats = jnp.array([rot.as_matrix() for rot in rots])
# XXX what the reference impl. calls the "precision matrix" is
# actually the square root of the precision matrix
precs = jnp.array(
    [rotmat @ np.diag(1 / extent) @ rotmat.T for (rotmat, extent) in zip(rotmats, extents)]
)
prec_chols = jnp.linalg.cholesky(precs)
ref_depth, ref_alpha, *_ = render_jit(means, prec_chols, log_weights, rays_trans, beta2, beta3)

In [ ]:
# GenMetaballs
camera = Intrinsics(fx=focal_length, fy=focal_length, cx=cx, cy=cy, width=width, height=height)
campose = geometry.Pose()

fmbs = [
    FMB(Pose.from_components(Rotation.from_quat(*rot.as_quat()), Vec3D(*mean)), *extent)
    for (mean, extent, rot) in zip(means, extents, rots)
]
scene = make_fmb_scene_from_values(fmbs, list(log_weights), device="gpu")

confidence = ZeroParameterConfidence()
blender = ThreeParameterBlender(beta1=beta3, beta2=beta2, eta=1.0)
image = render_fmbs(scene, blender, confidence, camera, campose)

img_view = image.as_view()
gmb_depth = img_view.depth.as_jax()
gmb_alpha = img_view.confidence.as_jax()

# gmb_depth = img_view.depth.as_jax()
# gmb_alpha = img_view.confidence.as_jax()

In [ ]:
visualize_results(gmb_depth, ref_depth.reshape(image_size), title="depth comparison")

In [ ]:
visualize_results(gmb_alpha, ref_alpha.reshape(image_size), title="alpha comparison")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

# Convert alpha to numpy and normalize
alpha_np = np.array(gmb_alpha)
alpha_normalized = (alpha_np - alpha_np.min()) / (alpha_np.max() - alpha_np.min())

# Apply colormap to get RGB image
colormap = matplotlib.colormaps.get_cmap(
    "viridis"
)  # You can use other colormaps like 'plasma', 'jet', etc.
alpha_image_rgb = colormap(alpha_normalized)[:, :, :3] * 255  # Remove alpha channel, keep RGB
alpha_image_rgb = alpha_image_rgb.astype(np.uint8)

In [ ]:
vec3d_to_np = lambda v: np.array([v.x, v.y, v.z])

import rerun as rr
from rerun.components import ViewCoordinates

rr.init("GenMetaBalls", spawn=False)
rr.connect_grpc()
rr.log("/", rr.Clear(recursive=True))

EXTEND_RAY_DIRS = 0.0

centers = np.array([[fmb.pose.tran.x, fmb.pose.tran.y, fmb.pose.tran.z] for fmb in fmbs])
extents = np.array([np.sqrt(fmb.extent) for fmb in fmbs])
quats = np.array([fmb.pose.rot.quat for fmb in fmbs])

cam_ray_dirs = []
for y in range(camera.height):
    for x in range(camera.width):
        cam_ray_dirs.append(vec3d_to_np(camera.get_ray_direction(x, y)))
cam_ray_dirs = np.array(cam_ray_dirs)
# cam_ray_dirs = camera_rays

# Generate random colors for each ellipsoid
np.random.seed(42)
colors = np.random.randint(0, 256, size=(len(fmbs), 3))

rr.log(
    "world/metaballs",
    rr.Ellipsoids3D(
        centers=centers,
        half_sizes=extents,
        quaternions=quats,
        colors=colors,
        fill_mode="Solid",
    ),
)

rr.log(
    "world/camera",
    rr.Pinhole(
        focal_length=camera.fx,
        principal_point=(camera.cx, camera.cy),
        width=camera.width,
        height=camera.height,
        image_plane_distance=1.0,
        camera_xyz=ViewCoordinates.RUB,
    ),
)
rr.log("world/camera", rr.Image(alpha_image_rgb))
rr.log(
    "world/rays",
    rr.Arrows3D(radii=0.01, colors=[128, 128, 128], vectors=cam_ray_dirs * EXTEND_RAY_DIRS),
)

# for ray_count in range(len(cam_ray_dirs)):
#     rr.set_time(timeline="ray_count", sequence=ray_count)
#     rr.log("world/rays", rr.Arrows3D(
#         radii=0.01,
#         colors=[128,128,128],
#         vectors=cam_ray_dirs[:ray_count] * EXTEND_RAY_DIRS))


In [ ]:
camera.get_ray_direction(0, 0), camera.get_ray_direction(0, 5)

In [ ]:
xx = np.ones((5, 5)).astype(np.uint8)
xx[3:] = 0
xx

In [ ]:
from genmetaballs._genmetaballs_bindings.image import CPUImage, GPUImage

yy = GPUImage(23, 37)

yy.as_view().depth